<a href="https://colab.research.google.com/github/Fares-pr0g/ML-journey-ep-2-Experimenting-with-NN-s-in-PyTorch/blob/main/Learning_PyTorch_08%3A%20Character_level_Language_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Character-level Language Modeling Using LSTM's

## DATASET: Let's work on the famous book The 48 Laws Of Power by Robert Greene

In [2]:
from google.colab import files

uploaded= files.upload()

Saving The 48 Laws Of Power.pdf to The 48 Laws Of Power.pdf


## Data Preprocessing:

In [3]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 10.3 MB/s eta 0:00:00


In [4]:
from pypdf import PdfReader

# turning the pdf book to a .txt file
pdf_path="/content/The 48 Laws Of Power.pdf"
reader= PdfReader(pdf_path)

text= ""

for page in reader.pages:
  text+= page.extract_text() + "\n"

In [5]:
print(len(text))
print(text[:2000])

1380195

THE 48 LAWS OF POWER

ROBERT GREENE has a degree in classical studies
and has been an editor at Esquire and oflmt magazines.
He is also a playwright and lives in Los Angeles.
JOOST ELFERS is the producer of 772: 48 Laws ofPawer
and also of
The Sam: Language cyFB:fi-tkdays
with Gary Goldschneider
The Semi Languages ofReiatiansiuyu
with Gary Goldschneider
Play with Hmr Facet‘
with Saxton Freymann

P
O
W
E
R
ROBERT GREENE
A JOOST ELFFERS PRODUCTION
P
PROFILE BOOKS

This paperback edition published in 2000
Reprinted 200}, 2002
First published in Great Britain in 1998 by
Profile Books Ltd
58A Hatnon Garden
London ECIN RLX
First published in the United States in 1998 by
Viking, a division of Penguin Putnam Inc.
Copyright ® Robert Greene aridjoost Elflers, 1998
A portion of this work first appeared in '17w Uzne Reader
Typeset in BE Baskerville
Printed and bound in Italy by
Legoprint S.p.a.
—
Lavis (TN)
The moral right of the authors has been asserted.
All rights reserved. Without lim

In [6]:
with open("48_laws_of_power.txt", "w") as f:
  f.write(text)

In [7]:
import numpy as np
# Reading and processing the text
# We don't need the acknowledgement section in for our data

with open("48_laws_of_power.txt", "r") as f:
  text= f.read()
end_indx= text.find(" As Man said, “When we\nfight you, we make sure you can’t get away.”\n")
text=text[:end_indx]
char_set= set(text)

print('Total Length:', len(text))
print("Unique Characters:", len(char_set))

Total Length: 1315513
Unique Characters: 110


In [8]:
# Encoding the text with a chat2int method
import numpy as np

chars_sorted= sorted(char_set)
char2int= {ch:i for i,ch in enumerate(chars_sorted)}
char_array= np.array(chars_sorted)
text_encoded= np.array([char2int[ch] for ch in text], dtype= np.int32)

# Let's run a small test
print('Text encoded shape:', text_encoded.shape)
print(text[:21], '== Encoding ==>', text_encoded[:21])
print(text_encoded[23:36], '== Decoding ==>', text[23:36])

Text encoded shape: (1315513,)

THE 48 LAWS OF POWER == Encoding ==> [ 0 53 41 38  1 21 25  1 45 34 56 52  1 48 39  1 49 48 56 38 51]
[51 48 35 38 51 53  1 40 51 38 38 47 38] == Decoding ==> ROBERT GREENE


In [9]:
import torch
from torch.utils.data import Dataset

seq_length= 40
chunk_size=seq_length+1
text_chunks= [text_encoded[i:i+chunk_size]
              for i in range(len(text_encoded)-chunk_size)]

class TextDataset(Dataset):

  def __init__(self, text_chunks):
    self.text_chunks= text_chunks

  def __len__(self):
    return len(self.text_chunks)

  def __getitem__(self, idx):
    text_chunk = self.text_chunks[idx]
    return text_chunk[:-1].long(), text_chunk[1:].long()

seq_dataset= TextDataset(torch.tensor(text_chunks))


/tmp/ipykernel_477/3309056888.py:21: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  seq_dataset= TextDataset(torch.tensor(text_chunks))


In [11]:
# Let's create a dataloader

from torch.utils.data import DataLoader

BATCH_SIZE= 64
torch.manual_seed(42)
seq_dl= DataLoader(seq_dataset, BATCH_SIZE, shuffle= True)


In [16]:
# test
from torch import nn
data_sample = next(iter(seq_dl))
data_sample

embedding= nn.Embedding(len(char_array),16)

embedding(data_sample[0]).shape

torch.Size([64, 40, 16])

### Device Agnostic Code:

In [10]:
import torch

device= "cuda" if torch.cuda.is_available() else 'cpu'
device

'cuda'

## Model 0: Unidirectional LSTM

In [19]:
import torch.nn as nn

class RNN (nn.Module):
  def __init__(self, vocab_size, embed_dim, rnn_hidden_size):
    super().__init__()
    self.embedding= nn.Embedding(vocab_size, embed_dim)
    self.rnn_hidden_size= rnn_hidden_size
    self.rnn= nn.LSTM(embed_dim, rnn_hidden_size, batch_first= True)
    self.fc= nn.Linear(rnn_hidden_size, vocab_size)

  def forward(self, X, hidden, cell):
    out=self.embedding(X).unsqueeze(1)
    out, (hidden, cell) = self.rnn(out, (hidden, cell))
    out= self.fc(out).reshape(out.size(0), -1)
    return out, hidden, cell

  def init_hidden(self, batch_size):
    hidden= torch.zeros(1, batch_size, self.rnn_hidden_size)
    cell= torch.zeros(1, batch_size, self.rnn_hidden_size)
    return hidden, cell


VOCAB_SIZE= len(char_array)
EMBED_DIM= 256
RNN_HIDDEN_SIZE= 512
torch.manual_seed(42)
model_0= RNN(VOCAB_SIZE, EMBED_DIM, RNN_HIDDEN_SIZE).to(device)
model_0


RNN(
  (embedding): Embedding(110, 256)
  (rnn): LSTM(256, 512, batch_first=True)
  (fc): Linear(in_features=512, out_features=110, bias=True)
)